In [1]:
import pandas as pd
import requests

print('pandas: ', pd.__version__)
print('requests: ', requests.__version__)

pandas:  3.0.5
requests:  2.34.2


## UN Comtrade API Exploration

### Goal

The goal of this study is to get acquainted with the UN Comtrade API. In this workflow, we will retrieve one international-trade response, inspect its JSON structure, identify the candidate grain, and save the raw response without transforming it.


### Source and Documentation

The project uses the public UN Comtrade API. Please follow the link to learn more: https://comtradeplus.un.org/TradeFlow.


### Request Design
#### Required Request Parameters
1. typeCode: Type of trade, C for commodities, S for service 
2. freqCode: Trade frequency, A for annual, M for monthly
3. clCode: Trade classification: HS/SS/BE/EB
    - HS: Physical Merchandise, uses standard 6-digit product codes, e.g., 090111 for coffee.
    - BE: Macroeconomic Categories, uses broad 3-digit economic codes, e.g. 111 for industrial food.
    - EB: Services Trade, uses service codes
    - SS: Historical/Socio-Economic, uses SITC codes used in older research.
#### Optional Query Parameters
- period: Month or year, e.g. 201002 or 2010, maximum 1
- reporterCode: M49 code of countries separated by comma
- partnerCode: M49 code of countries separated by comma
- cmdCode: Commodity code, maximum 1
- flowCode: Trade flow code, separated by comma

#### Example Request
- Endpoint: public preview final-data endpoint
- Product type: C — goods
- Frequency: A — annual
- Classification: HS — Harmonized System
- Period: 2023
- Reporter: 276 — Germany
- Partner: 0 — World
- Trade flow: X — exports
- Commodity: TOTAL — all commodities
- Authentication: no key required for preview
- Expected output: Germany's total merchandise exports to the World  for one annual period

GET TEMPLATE ENDPOINT:

`https://comtradeapi.un.org/public/v1/preview/{typeCode}/{freqCode}/{clCode}[?reporterCode][&period][&partnerCode][&cmdCode][&flowCode]`

Notice that typeCode, freqCode, clCode are included in path parameters, so they are required to form the endpoint URL.


### Request Execution



In [2]:
url = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
# set up query parameters
params = {
    "period": "2023",
    "reporterCode": "276",
    "cmdCode": "TOTAL",
    "flowCode": "X",
    "partnerCode": "0",
    "partner2Code": "0",
    "customsCode": "C00",
    "motCode": "0",
    "maxRecords": 500,
    "breakdownMode": "classic",
    "includeDesc": "true",
}
# get response
response = requests.get(
    url,
    params=params,
    timeout=30,
)
# check status, URL sent, respose preview. So far looks in order, response resembles a dict-like structure.
print("Status:", response.status_code)
print("Request URL:", response.url)
print("Response preview:")
print(response.text[:500])

Status: 200
Request URL: https://comtradeapi.un.org/public/v1/preview/C/A/HS?period=2023&reporterCode=276&cmdCode=TOTAL&flowCode=X&partnerCode=0&partner2Code=0&customsCode=C00&motCode=0&maxRecords=500&breakdownMode=classic&includeDesc=true
Response preview:
{"elapsedTime":"0.11 secs","count":1,"data":[{"typeCode":"C","freqCode":"A","refPeriodId":20230101,"refYear":2023,"refMonth":52,"period":"2023","reporterCode":276,"reporterISO":"DEU","reporterDesc":"Germany","flowCode":"X","flowDesc":"Export","partnerCode":0,"partnerISO":"W00","partnerDesc":"World","partner2Code":0,"partner2ISO":"W00","partner2Desc":"World","classificationCode":"H6","classificationSearchCode":"HS","isOriginalClassification":true,"cmdCode":"TOTAL","cmdDesc":"All Commodities","agg


### Response Structure



In [3]:
# check if response is successful
response.raise_for_status()
# parse data, this should get us a dictionary
payload = response.json()
# check the type and dictionary keys
print('Response type: ', type(payload))
print('Dictionary keys: ', payload.keys())

Response type:  <class 'dict'>
Dictionary keys:  dict_keys(['elapsedTime', 'count', 'data', 'error'])


In [4]:
# elapsedTime - how long API took to process request
# count - how many records are returned
# data - includes actual observations
# error - tells whether something went wrong
# check what type of values the keys include
payload_structure = {key: type(value) for key, value in payload.items()}
payload_structure

{'elapsedTime': str, 'count': int, 'data': list, 'error': str}

In [5]:
print("Error:", payload.get("error"))
print("Count:", payload.get("count"))
print("Elapsed time:", payload.get("elapsedTime"))

Error: 
Count: 1
Elapsed time: 0.11 secs


### Record Inspection



In [21]:
# separate the data
records = payload.get("data", [])

In [22]:
if not isinstance(records, list):
    raise TypeError("Expected payload['data'] to be a list.")

print("Number of records:", len(records))

Number of records: 1


In [ ]:
# inspect the first data point, in this case also the only one
if records:
    first_record = records[0]
    display(first_record)
else:
    print("No trade records were returned.")

{'typeCode': 'C',
 'freqCode': 'A',
 'refPeriodId': 20230101,
 'refYear': 2023,
 'refMonth': 52,
 'period': '2023',
 'reporterCode': 276,
 'reporterISO': 'DEU',
 'reporterDesc': 'Germany',
 'flowCode': 'X',
 'flowDesc': 'Export',
 'partnerCode': 0,
 'partnerISO': 'W00',
 'partnerDesc': 'World',
 'partner2Code': 0,
 'partner2ISO': 'W00',
 'partner2Desc': 'World',
 'classificationCode': 'H6',
 'classificationSearchCode': 'HS',
 'isOriginalClassification': True,
 'cmdCode': 'TOTAL',
 'cmdDesc': 'All Commodities',
 'aggrLevel': 0,
 'isLeaf': False,
 'customsCode': 'C00',
 'customsDesc': 'TOTAL CPC',
 'mosCode': '0',
 'motCode': 0,
 'motDesc': 'TOTAL MOT',
 'qtyUnitCode': -1,
 'qtyUnitAbbr': 'N/A',
 'qty': 0.0,
 'isQtyEstimated': False,
 'altQtyUnitCode': -1,
 'altQtyUnitAbbr': 'N/A',
 'altQty': 0.0,
 'isAltQtyEstimated': False,
 'netWgt': None,
 'isNetWgtEstimated': False,
 'grossWgt': 0.0,
 'isGrossWgtEstimated': False,
 'cifvalue': None,
 'fobvalue': 1725527288354.528,
 'primaryValue': 1

The returned record represents Germany's total annual merchandise exports to the World in 2023.

#### Main dimensions
- `typeCode = C`: commodity/goods trade
- `freqCode = A`: annual data
- `refYear = 2023` and `period = 2023`: reporting period
- `reporterCode = 276`, `reporterISO = DEU`: Germany
- `flowCode = X`: exports
- `partnerCode = 0`, `partnerDesc = World`: all partner countries
- `classificationCode = H6`: specific Harmonized System revision
- `cmdCode = TOTAL`: all commodities
- `aggrLevel = 0`: highest commodity aggregation
- `customsCode = C00`: all customs procedures
- `motCode = 0`: all modes of transport
#### Monetary measures
- `fobvalue`: export value excluding international freight and insurance
- `cifvalue`: import-style value including cost, insurance and freight
- `primaryValue`: main standardized trade-value field selected from the applicable CIF or FOB value
For this export record:
- `cifvalue` is missing
- `fobvalue` equals `primaryValue`
- `primaryValue` is approximately USD 1.726 trillion
#### Physical measures
- `qty` and `altQty` are unavailable at this aggregate level
- `netWgt` is missing
- `grossWgt` is recorded as zero, which should not yet be treated as a genuine measured zero
#### Quality flags
- quantity and weight estimation flags indicate whether values were estimated
- `isAggregate = True` shows this is an aggregated record
- `isReported = False` suggests this exact total row was derived rather than directly reported as a standalone observation
- `isOriginalClassification = True` indicates the original submitted commodity classification was used

In [25]:
if records:
    print(first_record.keys())

dict_keys(['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate'])


In [26]:
# the record doesn't have a nested object, but still it is common practive to normalize
records_df = pd.json_normalize(records)
print("Shape:", records_df.shape)
display(records_df.head())

Shape: (1, 47)


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,276,DEU,Germany,X,...,None,False,0.0,False,None,1.725527e+12,1.725527e+12,0,False,True


In [28]:
records_df.dtypes

typeCode                        str
freqCode                        str
refPeriodId                   int64
refYear                       int64
refMonth                      int64
period                          str
reporterCode                  int64
reporterISO                     str
reporterDesc                    str
flowCode                        str
flowDesc                        str
partnerCode                   int64
partnerISO                      str
partnerDesc                     str
partner2Code                  int64
partner2ISO                     str
partner2Desc                    str
classificationCode              str
classificationSearchCode        str
isOriginalClassification       bool
cmdCode                         str
cmdDesc                         str
aggrLevel                     int64
isLeaf                         bool
customsCode                     str
customsDesc                     str
mosCode                         str
motCode                     

In [31]:
missing_summary = (
    records_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)
missing_summary.head(5)

cifvalue    1
netWgt      1
typeCode    0
refYear     0
refMonth    0
dtype: int64

### Candidate Grain



One record appears to represent trade for:

reporter
× partner
× period
× trade flow
× commodity
× classification
× customs procedure
× transport mode

In [34]:
candidate_grain = [
    "typeCode",
    "freqCode",
    "period",
    "reporterCode",
    "partnerCode",
    "partner2Code",
    "flowCode",
    "classificationCode",
    "cmdCode",
    "customsCode",
    "motCode",
]

available_grain_columns = [
    column
    for column in candidate_grain
    if column in records_df.columns
]

missing_grain_columns = [
    column
    for column in candidate_grain
    if column not in records_df.columns
]

print("Available candidate columns:", available_grain_columns)
print("Missing candidate columns:", missing_grain_columns)

duplicate_count = records_df.duplicated(
    subset=available_grain_columns
).sum()

print("Candidate duplicates:", duplicate_count)

Available candidate columns: ['typeCode', 'freqCode', 'period', 'reporterCode', 'partnerCode', 'partner2Code', 'flowCode', 'classificationCode', 'cmdCode', 'customsCode', 'motCode']
Missing candidate columns: []
Candidate duplicates: 0


### Raw Data Storage


In [36]:
from pathlib import Path
Path.cwd()

WindowsPath('c:/Users/ozgur/GitHub/global-trade-flow-pipeline/notebooks')

In [ ]:
# find project root
def find_project_root(start_path: Path = Path.cwd()) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents,]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")
project_root = find_project_root()
project_root

WindowsPath('C:/Users/ozgur/GitHub/global-trade-flow-pipeline')

In [ ]:
import json
from datetime import datetime
# get current timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# create or select the raw data directory
raw_directory = (
    project_root
    / "data"
    / "raw"
)
raw_directory.mkdir(
    parents=True,
    exist_ok=True,
)
# create outputh file path
output_path = raw_directory / (
    f"comtrade_deu_world_exports_"
    f"2023_{timestamp}.json"
)

In [ ]:
# save raw response always, we may need to investigate source later and it should be available
with output_path.open("w", encoding="utf-8") as file:
    json.dump(
        payload,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved raw response to:")
print(output_path)

Saved raw response to:
C:\Users\ozgur\GitHub\global-trade-flow-pipeline\data\raw\comtrade_deu_world_exports_2023_20260802_223354.json


### Findings and Next Steps

#### Findings
- The public preview endpoint returned a successful JSON response.
- The top-level response contained:
  - `elapsedTime`: API processing time
  - `count`: number of returned records
  - `data`: list of trade records
  - `error`: API error message, empty when no error occurred
- Trade records were located under the `data` key.
- The request returned one aggregate record for Germany’s total exports to the World in 2023.
- The record contained:
  - coded and readable dimension fields
  - monetary trade values
  - physical quantity and weight fields
  - estimation and aggregation flags
- For this export record:
  - `fobvalue` equalled `primaryValue`
  - `cifvalue` was missing
  - quantity and weight measures were unavailable or not meaningful at the total-commodity aggregation level
- The complete source payload was saved under `data/raw/`.
### Candidate Grain
One record appears to represent a combination of:
reporter
× partner
× second partner
× period
× trade flow
× commodity
× classification
× customs procedure
× mode of transport
This grain is provisional because the current request returned only one
highly aggregated row and therefore we cannot fully validate uniqueness.
### Remaining Questions
- Which exact dimensions are required to guarantee uniqueness in larger responses?
- When does `partner2Code` differ from `partnerCode`?
- What is the exact meaning and use of `mosCode`?
- Which fields differ between classic and extended breakdown modes?
- Which quantity and weight measures are consistently available across years, commodities and flows?
- How should unavailable values represented as `None`, `0`, or `N/A` be interpreted during transformation?
- How should `isReported`, `isAggregate`, and the estimation flags be used in data-quality checks?
- How should authenticated extraction handle larger result sets and API limits?
### Next Steps
Move the exploratory request into a reusable extraction function with:
- configurable request parameters
- status-code validation
- timeout handling
- response-structure validation
- clear error messages
- timestamped raw JSON output
- basic logging of request metadata

In [14]:
## Reusable Extraction

import sys
from pathlib import Path

print(Path.cwd())

project_root = Path.cwd().parent # since e are at notebooks folder, we need to go up

# When importing something, search my project root first.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.extract import (
    fetch_trade_data,
    save_raw_payload,
)

c:\Users\ozgur\GitHub\global-trade-flow-pipeline\notebooks


In [15]:
# Test a Germany request
germany_payload = fetch_trade_data(
    type_code="C",
    frequency_code="A",
    classification_code="HS",
    period="2023",
    reporter_code="276",
    partner_code="0",
    commodity_code="TOTAL",
    flow_code="X",
)
# inspect the response
print("Count:", germany_payload["count"])
print("Error:", germany_payload["error"])

Count: 1
Error: 


In [ ]:
output_path = save_raw_payload(
    germany_payload,
    output_directory=project_root / "data" / "raw",
    reporter_code="276",
    partner_code="0",
    flow_code="X",
    period="2023",
)
print(output_path)

Saved raw response to:
c:\Users\ozgur\GitHub\global-trade-flow-pipeline\data\raw\comtrade_reporter-276_partner-0_flow-X_period-2023_20260803_210712.json
None


In [17]:
# Test a Turkiye request
turkiye_payload = fetch_trade_data(
    type_code="C",
    frequency_code="A",
    classification_code="HS",
    period="2023",
    reporter_code="792",
    partner_code="0",
    commodity_code="TOTAL",
    flow_code="X",
)
# inspect the response
turkiye_record = turkiye_payload["data"][0]
print(
    "Reporter:",
    turkiye_record["reporterDesc"],
)
print(
    "Primary value:",
    turkiye_record["primaryValue"],
)

Reporter: Türkiye
Primary value: 255627429011.0


In [ ]:
# Test multiple monthly requests
import time

monthly_records = []

for month in ["202301", "202302", "202303"]:
    payload = fetch_trade_data(
        type_code="C",
        frequency_code="M",
        classification_code="HS",
        period=month,
        reporter_code="276",
        partner_code="0",
        commodity_code="TOTAL",
        flow_code="X",
    )

    monthly_records.extend(payload["data"])
    time.sleep(1.1) # if we didn't have this, we would get 429 Too Many Requests

In [ ]:
monthly_records[0]

{'typeCode': 'C',
 'freqCode': 'M',
 'refPeriodId': 20230101,
 'refYear': 2023,
 'refMonth': 1,
 'period': '202301',
 'reporterCode': 276,
 'reporterISO': 'DEU',
 'reporterDesc': 'Germany',
 'flowCode': 'X',
 'flowDesc': 'Export',
 'partnerCode': 0,
 'partnerISO': 'W00',
 'partnerDesc': 'World',
 'partner2Code': 0,
 'partner2ISO': 'W00',
 'partner2Desc': 'World',
 'classificationCode': 'H6',
 'classificationSearchCode': 'HS',
 'isOriginalClassification': True,
 'cmdCode': 'TOTAL',
 'cmdDesc': 'All Commodities',
 'aggrLevel': 0,
 'isLeaf': False,
 'customsCode': 'C00',
 'customsDesc': 'TOTAL CPC',
 'mosCode': '0',
 'motCode': 0,
 'motDesc': 'TOTAL MOT',
 'qtyUnitCode': -1,
 'qtyUnitAbbr': 'N/A',
 'qty': 0.0,
 'isQtyEstimated': False,
 'altQtyUnitCode': -1,
 'altQtyUnitAbbr': 'N/A',
 'altQty': 0.0,
 'isAltQtyEstimated': False,
 'netWgt': None,
 'isNetWgtEstimated': False,
 'grossWgt': 0.0,
 'isGrossWgtEstimated': False,
 'cifvalue': None,
 'fobvalue': 139553698569.058,
 'primaryValue': 1

In [20]:
monthly_df = pd.json_normalize(monthly_records)
monthly_df[["period", "reporterDesc", "flowDesc", "primaryValue"]]

,period,reporterDesc,flowDesc,primaryValue
0,202301,Germany,Export,1.395537e+11
1,202302,Germany,Export,1.422445e+11
2,202303,Germany,Export,1.633963e+11
